In [ ]:
import cv2
import numpy as np

def load_image_preserve_alpha(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError("Failed to load image")
    return img


In [ ]:
def has_alpha(img):
    return len(img.shape) == 3 and img.shape[2] == 4


In [ ]:
path = "../photos/rotated_90.png"
img = load_image_preserve_alpha(path)
print(has_alpha(img))
len(img.shape)

In [5]:
def remove_alpha(img, bg_color=(255, 255, 255)):
    if not has_alpha(img):
        return img

    b, g, r, a = cv2.split(img)
    alpha = a.astype(float) / 255.0

    bg = np.ones_like(img[:, :, :3], dtype=float)
    bg[:, :, 0] *= bg_color[0]
    bg[:, :, 1] *= bg_color[1]
    bg[:, :, 2] *= bg_color[2]

    fg = cv2.merge([b, g, r]).astype(float)

    out = fg * alpha[..., None] + bg * (1 - alpha[..., None])
    return out.astype(np.uint8)


In [6]:
def ensure_rgb(img):
    if img.shape[2] == 3:
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


In [7]:
def validate_image(img):
    assert img.dtype == np.uint8
    assert len(img.shape) == 3
    assert img.shape[2] == 3  # no alpha


In [8]:
def normalize_uploaded_image(path):
    img = load_image_preserve_alpha(path)
    img = remove_alpha(img, bg_color=(255, 255, 255))
    img = ensure_rgb(img)
    validate_image(img)
    return img


In [9]:
path = "../photos/rotated_90.png"

img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
cv2.imwrite("before.jpg", img)  # likely has alpha issue

fixed = normalize_uploaded_image(path)
cv2.imwrite("after.jpg", fixed)


True